# Investigate Amazon Polarity and Run a TF-IDF + Logistic Regression Baseline

This notebook inspects the raw Amazon Polarity dataset already saved in the repo, samples the train/test splits for lightweight EDA, and ends with the reusable CLI and predictor usage snippets.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from textwrap import dedent

import pandas as pd
import plotly.express as px
from sklearn.feature_extraction.text import CountVectorizer

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "data" / "raw" / "amazon_polarity" / "dataset_dict.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from eWOM.sentiment_analysis.dataset_loader import AmazonPolarityLoader
from eWOM.sentiment_analysis.preprocess import SentimentPreprocessor

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "amazon_polarity"
TRAIN_SAMPLE = 50_000
TEST_SAMPLE = 10_000
RANDOM_STATE = 42

loader = AmazonPolarityLoader(DATA_DIR, random_state=RANDOM_STATE)
preprocessor = SentimentPreprocessor()

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset dir: {DATA_DIR}")
print(f"Train sample size: {TRAIN_SAMPLE:,}")
print(f"Test sample size: {TEST_SAMPLE:,}")

In [ ]:
dataset_info = loader.load_dataset_info("train")

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "num_examples": split_meta["num_examples"],
            "num_bytes": split_meta["num_bytes"],
        }
        for split_name, split_meta in dataset_info["splits"].items()
    ]
)

schema_summary = pd.DataFrame(
    [
        {"column": column_name, "dtype": column_meta["dtype"]}
        for column_name, column_meta in dataset_info["features"].items()
    ]
)

display(split_summary)
display(schema_summary)

In [ ]:
train_df = preprocessor.transform(loader.load_split("train", max_rows=TRAIN_SAMPLE))
test_df = preprocessor.transform(loader.load_split("test", max_rows=TEST_SAMPLE))

sample_summary = pd.DataFrame(
    [
        {
            "split": "train_sample",
            "rows": len(train_df),
            "positive_pct": round((train_df["label"] == 1).mean() * 100, 2),
            "negative_pct": round((train_df["label"] == 0).mean() * 100, 2),
        },
        {
            "split": "test_sample",
            "rows": len(test_df),
            "positive_pct": round((test_df["label"] == 1).mean() * 100, 2),
            "negative_pct": round((test_df["label"] == 0).mean() * 100, 2),
        },
    ]
)

display(sample_summary)
display(train_df.head(10))

In [ ]:
balance_df = pd.concat(
    [
        train_df.assign(split="train_sample"),
        test_df.assign(split="test_sample"),
    ],
    ignore_index=True,
)[["split", "label_text"]]

balance_plot = px.histogram(
    balance_df,
    x="label_text",
    color="label_text",
    facet_col="split",
    category_orders={"label_text": ["negative", "positive"]},
    title="Label balance across the sampled splits",
)
balance_plot.update_layout(showlegend=False)
balance_plot.show()

In [ ]:
def add_text_stats(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    stats = frame.copy()
    stats["split"] = split_name
    stats["char_len"] = stats["text"].str.len()
    stats["word_len"] = stats["text"].str.split().map(len)
    return stats

length_df = pd.concat(
    [
        add_text_stats(train_df, "train_sample"),
        add_text_stats(test_df, "test_sample"),
    ],
    ignore_index=True,
)

display(
    length_df.groupby(["split", "label_text"])[["word_len", "char_len"]]
    .median()
    .reset_index()
)

word_plot = px.histogram(
    length_df,
    x="word_len",
    color="label_text",
    facet_row="split",
    nbins=60,
    opacity=0.65,
    barmode="overlay",
    title="Word-count distributions by sentiment",
)
word_plot.show()

In [ ]:
def sample_reviews(frame: pd.DataFrame, label_text: str, n: int = 3) -> pd.DataFrame:
    return (
        frame.loc[frame["label_text"] == label_text, ["label_text", "text"]]
        .head(n)
        .reset_index(drop=True)
    )

display(sample_reviews(train_df, "positive", n=3))
display(sample_reviews(train_df, "negative", n=3))

In [ ]:
def top_terms_by_class(frame: pd.DataFrame, ngram_range: tuple[int, int], top_n: int = 15) -> pd.DataFrame:
    rows = []
    for label_text in ["negative", "positive"]:
        subset = frame.loc[frame["label_text"] == label_text, "text"]
        vectorizer = CountVectorizer(
            ngram_range=ngram_range,
            stop_words="english",
            min_df=5,
        )
        matrix = vectorizer.fit_transform(subset)
        counts = matrix.sum(axis=0).A1
        terms = vectorizer.get_feature_names_out()
        top_indices = counts.argsort()[::-1][:top_n]
        rows.extend(
            {
                "label_text": label_text,
                "term": terms[idx],
                "count": int(counts[idx]),
                "ngram": f"{ngram_range[0]}-{ngram_range[1]}",
            }
            for idx in top_indices
        )
    return pd.DataFrame(rows)

top_unigrams = top_terms_by_class(train_df, (1, 1))
top_bigrams = top_terms_by_class(train_df, (2, 2))

display(top_unigrams)
display(top_bigrams)

In [ ]:
run_instructions = dedent(
    """
    Edit the constants in `eWOM/sentiment_analysis/run_sentiment_pipeline.py` if you want to change
    paths, row caps, or TF-IDF settings, then run:

    python -m eWOM.sentiment_analysis.run_sentiment_pipeline
    """
)
print(run_instructions)


In [ ]:
predictor_example = dedent(
    """
    from eWOM.sentiment_analysis.predictor import SentimentPredictor

    predictor = SentimentPredictor(
        "models/sentiment/amazon_polarity_baseline.joblib",
        "models/sentiment/amazon_polarity_baseline_feature_builder.joblib",
    )
    prediction = predictor.predict_one(
        "This product arrived quickly and works great."
    )
    prediction
    """
)
print(predictor_example)